# Hyperspectral ViT — test & score

Loads a checkpoint written by `training.ipynb`, runs shape checks, reconstruction loss, and optional band-wise error plot.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_ROOT = Path("/workspace/Hyperspectral_Coolstuff/encoder_refactor").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from utilities import (
    band_indices_to_mask,
    band_masked_l1,
    load_checkpoint,
    masked_band_mse,
)
from model_definition import build_model, model_config_from_dict, describe_components
from spectral_mae import build_spectral_mae_model, spectral_mae_config_from_dict

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = _ROOT / "checkpoints" / "spectral_mae_latest.pt"

In [ ]:
assert CHECKPOINT_PATH.is_file(), f"Missing checkpoint: {CHECKPOINT_PATH}. Run training.ipynb first."

ckpt = load_checkpoint(CHECKPOINT_PATH, map_location=DEVICE)
mf = (ckpt.get("extra") or {}).get("model_family", "vit_reconstruction")

if mf == "spectral_mae":
    cfg = spectral_mae_config_from_dict(ckpt["model_config"])
    model = build_spectral_mae_model(cfg).to(DEVICE)
else:
    cfg = model_config_from_dict(ckpt["model_config"])
    model = build_model(cfg).to(DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("epoch", ckpt.get("epoch"), "extra", ckpt.get("extra"))
print(cfg)
if mf != "spectral_mae":
    for spec in describe_components(model):
        print(spec)

In [ ]:
@torch.no_grad()
def smoke_shapes(batch_size: int = 2):
    h, w = cfg.spatial_size
    c = cfg.in_channels
    x = torch.randn(batch_size, c, h, w, device=DEVICE)
    if mf == "spectral_mae":
        wl = torch.linspace(400.0, 2500.0, c, device=DEVICE)
        out = model(x, wavelengths_nm=wl)
        rec = out.cube_reconstruction
        assert rec.shape == x.shape
        bm = band_indices_to_mask(c, out.mask_band_indices, device=DEVICE)
        assert bm.sum().item() > 0
        print("OK: spectral MAE rec", tuple(rec.shape), "masked bands", int(bm.sum().item()))
    else:
        out = model(x)
        enc = out.tokens_encoded
        assert enc.ndim == 3 and enc.shape[0] == batch_size
        assert enc.shape[-1] == cfg.embed_dim
        rec = out.cube_reconstruction
        assert rec is not None and rec.shape == x.shape
        print("OK: encoded", tuple(enc.shape), "reconstruction", tuple(rec.shape))


smoke_shapes()

In [ ]:
@torch.no_grad()
def score_random_batch():
    h, w = cfg.spatial_size
    c = cfg.in_channels
    x = torch.randn(4, c, h, w, device=DEVICE)
    valid = torch.ones_like(x)
    if mf == "spectral_mae":
        wl = torch.linspace(400.0, 2500.0, c, device=DEVICE)
        out = model(x, wavelengths_nm=wl)
        pred = out.cube_reconstruction
        bm = band_indices_to_mask(c, out.mask_band_indices, device=DEVICE)
        loss = band_masked_l1(pred, x, valid, bm)
        print("band_masked_l1 on MAE-masked bands (random batch)", loss.item())
    else:
        out = model(x)
        pred = out.cube_reconstruction
        loss = masked_band_mse(pred, x, valid)
        print("masked_band_mse (random noise target)", loss.item())


score_random_batch()

In [ ]:
@torch.no_grad()
def per_band_mse(pred: torch.Tensor, target: torch.Tensor, valid: torch.Tensor):
    diff2 = (pred - target) ** 2 * valid
    denom = valid.sum(dim=(0, 2, 3)).clamp_min(1e-6)
    return (diff2.sum(dim=(0, 2, 3)) / denom).cpu().numpy()


h, w = cfg.spatial_size
c = cfg.in_channels
x = torch.randn(2, c, h, w, device=DEVICE)
valid = torch.ones_like(x)
if mf == "spectral_mae":
    wl = torch.linspace(400.0, 2500.0, c, device=DEVICE)
    out = model(x, wavelengths_nm=wl)
else:
    out = model(x)
pred = out.cube_reconstruction
pb = per_band_mse(pred, x, valid)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(pb)
ax.set_xlabel("band index")
ax.set_ylabel("MSE")
ax.set_title("Per-band reconstruction MSE (random batch)")
plt.tight_layout()
plt.show()